# Advanced: Multi-Strategy Numeric Analysis with PAMOLA.CORE

**Goal**: Master all 4 processing strategies for large-scale numeric analysis using operation.execute()

**What you'll learn:**
- **Strategy 1**: Standard processing (baseline performance)
- **Strategy 2**: Chunked processing (memory-efficient for large datasets)
- **Strategy 3**: Dask parallel processing (distributed computing)
- **Strategy 4**: Joblib parallel processing (multi-core optimization)
- Compare performance across processing methods
- Advanced statistical analysis and visualization
- Production deployment patterns

**Prerequisites:**
- Completed the simple notebook
- Understanding of statistical concepts
- Familiarity with operation.execute() pattern
- Python 3.8+
- PAMOLA.CORE installed (auto-installs if needed)

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── profiling/
        └── 02_numeric_analyzer_advanced.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import json
import sys
import os
from pathlib import Path
from datetime import datetime
import time
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print("   Repository: https://github.com/DGT-Network/PAMOLA.git")
        print("   Branch: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.profiling.analyzers.numeric import NumericOperation
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Complex Dataset

**How to use:**
- Run to load or generate 1000-record dataset
- Auto-creates sample if file not found
- Review data structure before proceeding

**What you'll see:**
- Load status (from file or generated)
- Dataset overview (1000 records, 8 columns)
- Sample records (first 5 rows)
- Column statistics (types, ranges, unique counts)

**Note:** Generated data auto-saves to examples/data_examples/ for reuse

In [ ]:
# Try to load from file first (in examples directory)
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'numeric_complex_data.csv'
print(f"📂 Looking for data at: {data_path}\n")

if data_path.exists():
    print(f"📂 Loading data from: {data_path}")
    df = read_csv(data_path)
    print(f"✓ Loaded {len(df)} records from file")
else:
    print("📊 Generating synthetic dataset (1000 records)...\n")
    
    np.random.seed(42)
    n_records = 1000
    
    # Generate complex numeric data with various distributions
    df = pd.DataFrame({
        'record_id': range(1, n_records + 1),
        'salary': np.random.normal(75000, 25000, n_records).round(2),
        'age': np.random.randint(18, 70, n_records),
        'experience_years': np.random.randint(0, 45, n_records),
        'performance_score': np.random.uniform(1.0, 5.0, n_records).round(2),
        'project_count': np.random.poisson(8, n_records),
        'satisfaction_rating': np.random.choice([1, 2, 3, 4, 5], n_records, p=[0.05, 0.10, 0.20, 0.35, 0.30]),
        'bonus_amount': np.random.exponential(5000, n_records).round(2)
    })
    
    # Add outliers
    df.loc[np.random.choice(n_records, 10, replace=False), 'salary'] = np.random.uniform(200000, 300000, 10)
    df.loc[np.random.choice(n_records, 5, replace=False), 'salary'] = np.random.uniform(10000, 20000, 5)
    
    # Add some null values
    null_indices = np.random.choice(n_records, 20, replace=False)
    df.loc[null_indices, 'performance_score'] = np.nan
    df.loc[np.random.choice(n_records, 15, replace=False), 'bonus_amount'] = np.nan
    
    # Save for future use (with error handling)
    try:
        data_path.parent.mkdir(parents=True, exist_ok=True)
        df.to_csv(data_path, index=False)
        print(f"✓ Generated and saved to: {data_path}")
    except PermissionError:
        print(f"⚠️  Cannot save to {data_path} (file may be open)")
        print(f"   Dataset generated in memory only")

print(f"\n📊 Dataset Overview:")
print("=" * 80)
print(f"  Records: {len(df):,}")
print(f"  Columns: {', '.join(df.columns)}")

print(f"\n🔍 Sample Records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_numeric_dtype(df[col]):
        print(f"  {col:20s} ({dtype_str:10s}): min={df[col].min():.2f}, max={df[col].max():.2f}, mean={df[col].mean():.2f}")
    else:
        print(f"  {col:20s} ({dtype_str:10s}): {df[col].nunique()} unique values")

print(f"\n💡 Perfect dataset for testing all 4 processing strategies!")

## Step 3: Setup Shared Environment

**How to use:**
1. **CUSTOMIZE FIELD_CONFIG**: Edit field names for each strategy
   - `"strategy1_field"`: For standard processing (default: "salary")
   - `"strategy2_field"`: For chunked processing (default: "salary")
   - `"strategy3_field"`: For Dask processing (default: "salary")
   - `"strategy4_field"`: For Joblib processing (default: "salary")
2. Run to validate fields and setup environment

**What you'll see:**
- Field validation (✓ marks for each field)
- Data type and range for each field
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and config ready (✓)

**Note:** All fields must exist in dataset and be numeric type before executing strategies

In [ ]:
# Field configuration - CUSTOMIZE THESE!
FIELD_CONFIG = {
    "strategy1_field": "salary",   # Standard processing
    "strategy2_field": "salary",   # Chunked processing
    "strategy3_field": "salary",   # Dask parallel
    "strategy4_field": "salary"    # Joblib parallel
}

# Validate all fields exist in dataset
print("📋 Validating field configuration...\n")
for strategy, field_name in FIELD_CONFIG.items():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' for {strategy} not found!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update FIELD_CONFIG"
        )
    
    # Validate numeric type
    if not pd.api.types.is_numeric_dtype(df[field_name]):
        raise ValueError(
            f"❌ Field '{field_name}' is not numeric type!\n"
            f"Type: {df[field_name].dtype}\n"
            f"Please choose a numeric field"
        )
    
    print(f"  ✓ {strategy:20s}: '{field_name}'")
    print(f"    Type: {df[field_name].dtype}")
    print(f"    Range: {df[field_name].min():.2f} to {df[field_name].max():.2f}")
    print(f"    Mean: {df[field_name].mean():.2f}, Median: {df[field_name].median():.2f}")

# Configuration helper (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset"
    }

# Create task directory (in examples/data_examples)
examples_dir = project_root / 'examples'
task_dir = examples_dir / 'data_examples' / 'advanced_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="advanced_001",
    task_type="multi_strategy",
    description="Multi-strategy numeric field analysis",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

# Get config
kwargs = create_config_kwargs()
print(f"✓ Config kwargs ready")

# Create DataSource
data_source = DataSource(
    dataframes={"main_dataset": df}
)
print("✓ DataSource created")

print(f"\n✅ Shared environment ready for all strategies!")

## STRATEGY 1: Standard Processing

**How to use:**
- Baseline processing without optimization
- Single-threaded, in-memory analysis
- Run to execute standard strategy

**What you'll see:**
- Configuration summary
- Progress: validation → processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `bins=20`: Number of histogram bins
- `detect_outliers=True`: IQR-based outlier detection
- `test_normality=True`: Statistical normality tests
- `use_vectorization=False`: No parallel processing
- `use_dask=False`: No distributed computing

**Note:** Provides baseline performance for comparison with optimized strategies

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 1: STANDARD PROCESSING")
print("=" * 80 + "\n")

# Setup progress tracker
tracker_s1 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 1: Standard",
    unit="steps",
    track_memory=True,
    level=0
)

# Create operation
operation_s1 = NumericOperation(
    # Core parameters
    field_name=FIELD_CONFIG["strategy1_field"],
    bins=20,
    detect_outliers=True,
    test_normality=True,
    near_zero_threshold=1e-10,
    
    # Output configuration
    output_format='json',
    profile_type='numeric',
    
    # Processing settings (STANDARD - no optimization)
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    chunk_size=10000,
    use_dask=False,
    npartitions=None,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 1 configured")
print(f"\n⚙️  Executing...\n")
start_time = time.time()

# Execute
result_s1 = operation_s1.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy1_standard',
    reporter=task_reporter,
    progress_tracker=tracker_s1,
    **kwargs
)

elapsed_s1 = time.time() - start_time
print(f"\n✅ Strategy 1 completed in {elapsed_s1:.2f}s")

# Load results (NEWEST file)
output_files_s1 = sorted(
    list((task_dir / 'strategy1_standard' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s1:
    with open(output_files_s1[0], 'r') as f:
        result_data_s1 = json.load(f)
    field_s1 = FIELD_CONFIG["strategy1_field"]
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid values: {result_data_s1.get('valid_count', 0):,}")
    print(f"   Null values: {result_data_s1.get('null_count', 0):,} ({result_data_s1.get('null_percentage', 0):.2f}%)")
    stats = result_data_s1.get('stats', {})
    if stats:
        print(f"   Mean: {stats.get('mean', 0):.2f}")
        print(f"   Std Dev: {stats.get('std', 0):.2f}")
        if 'outliers' in stats:
            print(f"   Outliers: {stats['outliers'].get('count', 0):,}")

## STRATEGY 2: Chunked Processing

**How to use:**
- Memory-efficient processing for large datasets
- Processes data in manageable chunks
- Run to execute chunked strategy

**What you'll see:**
- Configuration summary
- Progress: validation → chunked processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `chunk_size=250`: Process 250 rows at a time
- `use_vectorization=False`: Sequential chunk processing
- `use_dask=False`: No distributed computing
- Lower memory footprint than Strategy 1

**Note:** Best for datasets that don't fit in memory or when memory is constrained

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 2: CHUNKED PROCESSING")
print("=" * 80 + "\n")

tracker_s2 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 2: Chunked",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s2 = NumericOperation(
    field_name=FIELD_CONFIG["strategy2_field"],
    bins=20,
    detect_outliers=True,
    test_normality=True,
    
    # CHUNKED processing settings
    chunk_size=250,  # Small chunks for demonstration
    use_vectorization=False,
    use_dask=False,
    
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 2 configured")
print(f"   Chunk size: {operation_s2.chunk_size} rows")
print(f"   Total chunks: {len(df) // operation_s2.chunk_size + 1}")
start_time = time.time()

result_s2 = operation_s2.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy2_chunked',
    reporter=task_reporter,
    progress_tracker=tracker_s2,
    **kwargs
)

elapsed_s2 = time.time() - start_time
print(f"\n✅ Strategy 2 completed in {elapsed_s2:.2f}s")

# Load results
output_files_s2 = sorted(
    list((task_dir / 'strategy2_chunked' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s2:
    with open(output_files_s2[0], 'r') as f:
        result_data_s2 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid values: {result_data_s2.get('valid_count', 0):,}")
    stats = result_data_s2.get('stats', {})
    if stats:
        print(f"   Mean: {stats.get('mean', 0):.2f}")
        print(f"   Memory efficient: ✓")

## STRATEGY 3: Dask Parallel Processing

**How to use:**
- Distributed computing with Dask
- Parallel processing across multiple partitions
- Run to execute Dask strategy

**What you'll see:**
- Configuration summary
- Progress: validation → parallel processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `use_dask=True`: Enable Dask distributed computing
- `npartitions=4`: Split data into 4 partitions
- `chunk_size=250`: Chunk size per partition
- Scales to very large datasets

**Note:** Best for very large datasets (10M+ rows) or distributed computing environments

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 3: DASK PARALLEL PROCESSING")
print("=" * 80 + "\n")

tracker_s3 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 3: Dask",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s3 = NumericOperation(
    field_name=FIELD_CONFIG["strategy3_field"],
    bins=20,
    detect_outliers=True,
    test_normality=True,
    
    # DASK parallel processing settings
    use_dask=True,
    npartitions=4,  # Number of parallel partitions
    chunk_size=250,
    use_vectorization=False,
    
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 3 configured")
print(f"   Partitions: {operation_s3.npartitions}")
print(f"   Parallel: Dask distributed")
start_time = time.time()

result_s3 = operation_s3.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy3_dask',
    reporter=task_reporter,
    progress_tracker=tracker_s3,
    **kwargs
)

elapsed_s3 = time.time() - start_time
print(f"\n✅ Strategy 3 completed in {elapsed_s3:.2f}s")

# Load results
output_files_s3 = sorted(
    list((task_dir / 'strategy3_dask' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s3:
    with open(output_files_s3[0], 'r') as f:
        result_data_s3 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid values: {result_data_s3.get('valid_count', 0):,}")
    stats = result_data_s3.get('stats', {})
    if stats:
        print(f"   Mean: {stats.get('mean', 0):.2f}")
        print(f"   Distributed: ✓")

## STRATEGY 4: Joblib Parallel Processing

**How to use:**
- Multi-core parallel processing with Joblib
- Optimal for single-machine parallelization
- Run to execute Joblib strategy

**What you'll see:**
- Configuration summary
- Progress: validation → parallel processing → metrics → viz → save
- Completion time and status
- Artifact count (4-6 files expected)

**Key parameters:**
- `use_vectorization=True`: Enable parallel processing
- `parallel_processes=4`: Use 4 CPU cores
- `chunk_size=250`: Chunk size per process
- Best single-machine performance

**Note:** Optimal for medium-to-large datasets on multi-core machines (not distributed)

In [ ]:
print("\n" + "=" * 80)
print("🔄 STRATEGY 4: JOBLIB PARALLEL PROCESSING")
print("=" * 80 + "\n")

tracker_s4 = HierarchicalProgressTracker(
    total=6,
    description="Strategy 4: Joblib",
    unit="steps",
    track_memory=True,
    level=0
)

operation_s4 = NumericOperation(
    field_name=FIELD_CONFIG["strategy4_field"],
    bins=20,
    detect_outliers=True,
    test_normality=True,
    
    # JOBLIB parallel processing settings
    use_vectorization=True,
    parallel_processes=4,  # Number of CPU cores
    chunk_size=250,
    use_dask=False,
    
    use_cache=False,
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Strategy 4 configured")
print(f"   Processes: {operation_s4.parallel_processes}")
print(f"   Parallel: Joblib multi-core")
start_time = time.time()

result_s4 = operation_s4.execute(
    data_source=data_source,
    task_dir=task_dir / 'strategy4_joblib',
    reporter=task_reporter,
    progress_tracker=tracker_s4,
    **kwargs
)

elapsed_s4 = time.time() - start_time
print(f"\n✅ Strategy 4 completed in {elapsed_s4:.2f}s")

# Load results
output_files_s4 = sorted(
    list((task_dir / 'strategy4_joblib' / 'output').glob('*.json')),
    key=lambda x: x.stat().st_mtime, reverse=True
)
if output_files_s4:
    with open(output_files_s4[0], 'r') as f:
        result_data_s4 = json.load(f)
    
    print(f"\n📊 Analysis Results:")
    print(f"   Valid values: {result_data_s4.get('valid_count', 0):,}")
    stats = result_data_s4.get('stats', {})
    if stats:
        print(f"   Mean: {stats.get('mean', 0):.2f}")
        print(f"   Multi-core: ✓")

## Step 5: Strategy Comparison

**How to use:**
- Run AFTER all strategies complete
- Review performance and effectiveness

**What you'll see:**
- Execution times (fastest to slowest)
- Total processing time
- Speedup factors vs baseline
- Strategy rankings by performance

**Note:** Fastest strategy depends on dataset size, hardware, and available cores

In [ ]:
print("\n" + "=" * 80)
print("📊 STRATEGY COMPARISON")
print("=" * 80 + "\n")

print("⏱️  Execution Time:")
print(f"  Strategy 1 (Standard): {elapsed_s1:6.2f}s  [baseline]")
print(f"  Strategy 2 (Chunked):  {elapsed_s2:6.2f}s  [{elapsed_s1/elapsed_s2:.2f}x]")
print(f"  Strategy 3 (Dask):     {elapsed_s3:6.2f}s  [{elapsed_s1/elapsed_s3:.2f}x]")
print(f"  Strategy 4 (Joblib):   {elapsed_s4:6.2f}s  [{elapsed_s1/elapsed_s4:.2f}x]")
print(f"  Total:                 {elapsed_s1 + elapsed_s2 + elapsed_s3 + elapsed_s4:6.2f}s")

# Find fastest strategy
strategies = [
    ('Standard', elapsed_s1),
    ('Chunked', elapsed_s2),
    ('Dask', elapsed_s3),
    ('Joblib', elapsed_s4)
]
strategies.sort(key=lambda x: x[1])

print(f"\n🏆 Performance Ranking:")
for i, (name, time_val) in enumerate(strategies, 1):
    print(f"  {i}. {name:12s} - {time_val:.2f}s")

## Step 6: Detailed Artifact Review

**How to use:**
- Run for comprehensive artifact inspection
- Focus on validation points per strategy

**What you'll see (per strategy):**
1. **Metrics JSON**: Statistical metrics, performance data
2. **Output Data**: Analysis results with statistics
3. **Visualizations**: Histogram, boxplot, Q-Q plot (latest batch only)

**Note:** Only newest files shown to avoid confusion from multiple runs

In [ ]:
# Review all strategies
strategy_dirs = [
    ('strategy1_standard', 'Strategy 1: Standard Processing'),
    ('strategy2_chunked', 'Strategy 2: Chunked Processing'),
    ('strategy3_dask', 'Strategy 3: Dask Parallel'),
    ('strategy4_joblib', 'Strategy 4: Joblib Parallel')
]

for dir_name, title in strategy_dirs:
    strategy_dir = task_dir / dir_name
    if not strategy_dir.exists():
        continue
        
    print("\n" + "=" * 80)
    print(f"📊 {title}")
    print("=" * 80)
    
    # 1. Metrics Summary (NEWEST - with filtering)
    metrics_dir = strategy_dir / 'metrics'
    if metrics_dir.exists():
        metrics_files = list(metrics_dir.glob('*.json'))
        
        if metrics_files:
            # Exclude data_types files
            filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

            if filtered:
                target_files = filtered
                print(f"\n✓ Found {len(filtered)} metrics file(s)")
            else:
                target_files = metrics_files
                print(f"\n⚠ Fallback to ALL {len(metrics_files)} file(s)")

            # Pick latest
            target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
            latest_metrics_file = target_files[0]
            
            print(f"📄 {latest_metrics_file.name}")
            print("   " + "-" * 60)
            
            try:
                with open(latest_metrics_file, 'r') as f:
                    data = json.load(f)
                    metrics = data.get('metrics', {})
                
                if 'performance' in metrics:
                    perf = metrics['performance']
                    print(f"   ⏱️  Performance:")
                    print(f"      Duration          : {perf.get('duration_seconds', 0):.4f}s")
                    print(f"      Memory used       : {perf.get('memory_used_mb', 0):.2f} MB")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
    
    # 2. Detailed Statistics (from output/)
    output_dir = strategy_dir / 'output'
    if output_dir and output_dir.exists():
        output_files = sorted(
            list(output_dir.glob('*.json')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if output_files:
            print(f"\n📊 STATISTICAL ANALYSIS: {output_files[0].name}")
            print("   " + "-" * 60)
            
            try:
                with open(output_files[0], 'r') as f:
                    output_data = json.load(f)
                
                # Basic counts
                print(f"   Total rows           : {output_data.get('total_rows', 0):,}")
                print(f"   Valid count          : {output_data.get('valid_count', 0):,}")
                print(f"   Null count           : {output_data.get('null_count', 0):,} ({output_data.get('null_percentage', 0):.2f}%)")
                
                # Detailed statistics
                if 'stats' in output_data:
                    stats = output_data['stats']
                    
                    # Descriptive statistics
                    print(f"\n   📈 Descriptive Statistics:")
                    print(f"      Min               : {stats.get('min', 0):,.2f}")
                    print(f"      Max               : {stats.get('max', 0):,.2f}")
                    print(f"      Mean              : {stats.get('mean', 0):,.2f}")
                    print(f"      Median            : {stats.get('median', 0):,.2f}")
                    print(f"      Std Dev           : {stats.get('std', 0):,.2f}")
                    print(f"      Variance          : {stats.get('variance', 0):,.2f}")
                    
                    # Distribution shape
                    print(f"\n   📊 Distribution Shape:")
                    print(f"      Skewness          : {stats.get('skewness', 0):.4f}")
                    print(f"      Kurtosis          : {stats.get('kurtosis', 0):.4f}")
                    
                    # Value composition
                    print(f"\n   🔢 Value Composition:")
                    print(f"      Zero count        : {stats.get('zero_count', 0):,} ({stats.get('zero_percentage', 0):.2f}%)")
                    print(f"      Negative count    : {stats.get('negative_count', 0):,} ({stats.get('negative_percentage', 0):.2f}%)")
                    print(f"      Positive count    : {stats.get('positive_count', 0):,} ({stats.get('positive_percentage', 0):.2f}%)")
                    
                    # Key percentiles
                    if 'percentiles' in stats:
                        percentiles = stats['percentiles']
                        print(f"\n   📉 Key Percentiles:")
                        key_pcts = ['p1', 'p5', 'p10', 'p25', 'p50', 'p75', 'p90', 'p95', 'p99']
                        for pct in key_pcts:
                            if pct in percentiles:
                                value = percentiles[pct]
                                print(f"      {pct:5s}            : {value:,.2f}")
                    
                    # Outliers analysis
                    if 'outliers' in stats:
                        outliers = stats['outliers']
                        print(f"\n   🎯 Outliers Analysis:")
                        print(f"      IQR               : {outliers.get('iqr', 0):,.2f}")
                        print(f"      Lower bound       : {outliers.get('lower_bound', 0):,.2f}")
                        print(f"      Upper bound       : {outliers.get('upper_bound', 0):,.2f}")
                        print(f"      Outlier count     : {outliers.get('count', 0):,} ({outliers.get('percentage', 0):.2f}%)")
                        
                        # Show sample outliers
                        if 'sample' in outliers and outliers['sample']:
                            print(f"      Sample outliers   : ", end="")
                            samples = [f"{x:,.2f}" for x in outliers['sample'][:5]]
                            print(", ".join(samples))
                            if len(outliers['sample']) > 5:
                                print(f"                          ... and {len(outliers['sample']) - 5} more")
                    
                    # Normality tests
                    if 'normality' in stats:
                        normality = stats['normality']
                        print(f"\n   🔬 Normality Tests:")
                        print(f"      Is Normal         : {normality.get('is_normal', False)}")
                        print(f"      Tests passed      : {normality.get('normal_test_passed', 0)}/{normality.get('normal_test_count', 0)}")
                        
                        # Shapiro-Wilk test
                        if 'shapiro' in normality:
                            shapiro = normality['shapiro']
                            print(f"      Shapiro-Wilk      : statistic={shapiro.get('statistic', 0):.4f}, p={shapiro.get('p_value', 0):.6f}")
                        
                        # Kolmogorov-Smirnov test
                        if 'ks' in normality:
                            ks = normality['ks']
                            print(f"      Kolmogorov-Smirnov: statistic={ks.get('statistic', 0):.4f}, p={ks.get('p_value', 0):.6f}")
                    
                    # Histogram summary
                    if 'histogram' in stats:
                        histogram = stats['histogram']
                        print(f"\n   📊 Distribution Histogram:")
                        print(f"      Bins              : {len(histogram.get('bins', []))}")
                        
                        # Show top 5 bins with highest counts
                        if 'bins' in histogram and 'counts' in histogram:
                            bins = histogram['bins']
                            counts = histogram['counts']
                            
                            # Combine and sort by count
                            bin_data = list(zip(bins, counts))
                            bin_data_sorted = sorted(bin_data, key=lambda x: x[1], reverse=True)
                            
                            print(f"      Top 5 bins:")
                            for bin_range, count in bin_data_sorted[:5]:
                                pct = (count / sum(counts) * 100) if sum(counts) > 0 else 0
                                print(f"        {bin_range:30s}: {count:4,} ({pct:5.2f}%)")
                
            except Exception as e:
                print(f"   ⚠️  Error: {e}")
                import traceback
                traceback.print_exc()
    
    # 3. Visualizations (NEWEST BATCH)
    viz_dir = strategy_dir / 'visualizations'
    if viz_dir.exists():
        viz_files = sorted(
            list(viz_dir.glob('*.png')),
            key=lambda x: x.stat().st_mtime, reverse=True
        )
        
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            latest_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < 10
            ]
            
            print(f"\n📊 VISUALIZATIONS: {len(latest_batch)} files")
            for viz in latest_batch[:2]:  # Show first 2 only
                print(f"\n   📈 {viz.stem.replace('_', ' ').title()}")
                try:
                    display(Image(filename=str(viz)))
                except:
                    print(f"      (Display error)")

## Step 7: Statistical Consistency Check

**How to use:**
- Run AFTER all strategies complete
- Verifies all strategies produce identical results

**What you'll see:**
- Mean values from each strategy
- Standard deviation comparison
- Consistency verification (all should match)
- Any discrepancies highlighted

**Note:** All strategies should produce identical statistical results despite different processing methods

In [ ]:
print("\n" + "=" * 80)
print("🔬 STATISTICAL CONSISTENCY CHECK")
print("=" * 80 + "\n")

# Collect statistics from all strategies
all_stats = []
strategy_names = ['Standard', 'Chunked', 'Dask', 'Joblib']

for i, data in enumerate([result_data_s1, result_data_s2, result_data_s3, result_data_s4], 1):
    stats = data.get('stats', {})
    all_stats.append({
        'strategy': strategy_names[i-1],
        'mean': stats.get('mean', 0),
        'std': stats.get('std', 0),
        'min': stats.get('min', 0),
        'max': stats.get('max', 0)
    })

print("📊 Statistical Results Comparison:")
print(f"\n{'Strategy':<12} {'Mean':>12} {'Std Dev':>12} {'Min':>12} {'Max':>12}")
print("-" * 60)
for stat in all_stats:
    print(f"{stat['strategy']:<12} {stat['mean']:>12.2f} {stat['std']:>12.2f} {stat['min']:>12.2f} {stat['max']:>12.2f}")

# Check consistency
means = [s['mean'] for s in all_stats]
stds = [s['std'] for s in all_stats]

mean_diff = max(means) - min(means)
std_diff = max(stds) - min(stds)

print(f"\n✅ Consistency Check:")
print(f"   Mean variance: {mean_diff:.6f}")
print(f"   Std Dev variance: {std_diff:.6f}")

if mean_diff < 0.01 and std_diff < 0.01:
    print(f"   Status: ✓ All strategies produce consistent results")
else:
    print(f"   Status: ⚠️  Some variance detected (may be due to rounding)")

## Step 8: Export Final Data

**How to use:**
- Run AFTER all strategies complete
- Exports analysis summary for each strategy

**What you'll see (per strategy):**
- Export confirmation with file path
- Key statistics summary
- Processing method and performance

**Output structure:**
```
advanced_output/
├── strategy1_standard/
│   └── output/*.json
├── strategy2_chunked/
│   └── output/*.json
├── strategy3_dask/
│   └── output/*.json
└── strategy4_joblib/
    └── output/*.json
```

**Note:** Each strategy folder contains complete analysis results with metrics and visualizations

In [ ]:
print("=" * 80)
print("📦 EXPORT SUMMARY")
print("=" * 80)

print(f"\n📂 All files saved to: {task_dir}")

strategies_info = [
    ('strategy1_standard', 'Standard Processing', elapsed_s1, result_data_s1),
    ('strategy2_chunked', 'Chunked Processing', elapsed_s2, result_data_s2),
    ('strategy3_dask', 'Dask Parallel', elapsed_s3, result_data_s3),
    ('strategy4_joblib', 'Joblib Parallel', elapsed_s4, result_data_s4)
]

print(f"\n📁 Strategy Summaries:")
for dir_name, strategy_name, elapsed, result_data in strategies_info:
    print(f"\n  {strategy_name}:")
    print(f"    Directory: {dir_name}/")
    print(f"    Processing time: {elapsed:.2f}s")
    print(f"    Valid records: {result_data.get('valid_count', 0):,}")
    stats = result_data.get('stats', {})
    if stats:
        print(f"    Mean: {stats.get('mean', 0):.2f}")
        print(f"    Std Dev: {stats.get('std', 0):.2f}")

total_time = sum([elapsed_s1, elapsed_s2, elapsed_s3, elapsed_s4])
print(f"\n⏱️  Total processing time: {total_time:.2f}s")

print("\n" + "=" * 80)
print("🎉 EXPORT COMPLETE!")
print("=" * 80)

## 🎯 Summary

**Accomplished:**
- ✅ 4 processing strategies implemented and compared
- ✅ Performance benchmarking completed
- ✅ Statistical consistency verified across strategies
- ✅ Production-ready artifacts generated

**Key takeaways:**
- Standard processing: Simple, reliable baseline
- Chunked processing: Memory-efficient for large datasets
- Dask parallel: Best for distributed/very large datasets
- Joblib parallel: Optimal single-machine multi-core performance

**Strategy recommendations:**
- **Use Standard** when: Dataset < 100K rows, simple analysis needed
- **Use Chunked** when: Memory constrained, dataset > 1M rows
- **Use Dask** when: Dataset > 10M rows, distributed environment available
- **Use Joblib** when: Dataset 100K-10M rows, multi-core machine available

**Next steps:**
- Test with your own datasets
- Tune parameters (chunk_size, npartitions, parallel_processes)
- Deploy to production environment
- Monitor performance metrics

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🐙 [GitHub Repository](https://github.com/DGT-Network/PAMOLA)